In [1]:
from collections import Counter, defaultdict
import json
import os
import re

from datasets import load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
dict_label_pattern = re.compile(r"\A\([^)]+\)") # rm labels like (Intransitive)
# [from 20th c. not cleaned!

In [3]:
out_dir = "periti_2025_data"
os.makedirs(out_dir, exist_ok=True)
ds = load_dataset("FrancescoPeriti/Dbnary-en")

In [4]:
def split_sft_dpo(split):
    ds = load_dataset("FrancescoPeriti/Dbnary-en", split=split)
    ds.set_format("pandas")
    ds = ds[:]
    print(ds.head())
    ds["definition"] = ds["definition"].apply(
        lambda x: re.sub(dict_label_pattern, "", x).strip()
    )
    print(ds.head())
    target_word = defaultdict(list)
    for word, word_df in ds.groupby("target"):
        target_word['words'].append(word)
        n_unique = min(21, len(word_df.definition.unique())) # exact values above 21 in dev happens only once, have to merge
        # for stratifying
        target_word['counts'].append(n_unique)
    print(Counter(target_word['counts']))
    target_word_df = pd.DataFrame(target_word)
    sft, dpo = train_test_split(target_word_df, random_state=42, stratify=target_word_df["counts"], train_size=0.5)
    print(f"Number of target words in SFT {split}: {sft.shape[0]}")
    print(f"Number of target words in DPO {split}: {dpo.shape[0]}")
    sft_df = ds[ds["target"].isin(sft.words)]
    print(f"Number of examples in SFT {split}: {sft_df.shape[0]}")
    print(sft_df.head())
    dpo_df = ds[ds["target"].isin(dpo.words)]
    print(f"Number of examples in DPO {split}: {dpo_df.shape[0]}")
    print(dpo_df.head())
    intersected_set = set(sft_df.target.unique()).intersection(set(dpo_df.target.unique()))
    print(intersected_set)
    assert not intersected_set
    sft_df.to_csv(os.path.join(out_dir, f"sft_{split}.tsv"), sep='\t')
    with open(os.path.join(out_dir, f"{split}_dpo.jsonl"), "w") as f:
        for _, record in dpo_df.iterrows():
            f.write(json.dumps(
                {
                    'target': record['target'],
                    'example': record['example'],
                    'definition': record['definition'],
                }
            ) + '\n'
            )